<a href="https://colab.research.google.com/github/mksmk2012/A-B-Test-Analysis/blob/main/Project_AB_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A/B Test Analysis: Statistical Significance

## Links
- [Tableau](https://public.tableau.com/views/Book1_17841370064470/Dashboard3?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)
- [Data Set](https://github.com/mksmk2012/AB_Test_Analysis/blob/main/results/ab_test_reslts%20-%20ab_test_reslts.csv)



## Project stages
1. **BigQuery** Session, event, account and order data are joined with the A/B test assignment table into one dataset: one row per date, country, device, continent, channel, test, test group and event.
2. **Python.** For each of the 4 tests and each of the 4 metrics (add_payment_info, add_shipping_info, begin_checkout, new_accounts), conversion rates of the control and experimental groups are compared with a two-proportion z-test.
3. **Tableau.** The results CSV is visualized on a dashboard with metric cards, a conversion rate comparison, a p-value chart and a test selector.

In [ ]:
import numpy as np
import pandas as pd
!pip install -q statsmodels
from statsmodels.stats.proportion import proportions_ztest
!pip install -q google-cloud-bigquery

In [ ]:
from google.colab import auth

auth.authenticate_user()

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project='data-analytics-mate')

In [ ]:
query = """
with session_info AS (
 SELECT
 s.date,
 s.ga_session_id,
 sp.country,
 sp.device,
 sp.continent,
 sp.channel,
 ab.test,
 ab.test_group,
 FROM `DA.ab_test` ab
 JOIN `DA.session` s
 ON ab.ga_session_id = s.ga_session_id
 JOIN `DA.session_params` sp
 ON sp.ga_session_id = ab.ga_session_id),


session_with_orders AS (
SELECT
 session_info.date,
 session_info.country,
 session_info.device,
 session_info.continent,
 session_info.channel,
 session_info.test,
 session_info.test_group,
 COUNT(DISTINCT o.ga_session_id) AS session_with_orders
FROM `DA.order` o
JOIN session_info
ON o.ga_session_id = session_info.ga_session_id
GROUP BY
 session_info.date,
 session_info.country,
 session_info.device,
 session_info.continent,
 session_info.channel,
 session_info.test,
 session_info.test_group),


events AS (
SELECT
session_info.date,
 session_info.country,
 session_info.device,
 session_info.continent,
 session_info.channel,
 session_info.test,
 session_info.test_group,
 ep.event_name,
 COUNT(ep.ga_session_id) AS event_cnt
FROM `DA.event_params` ep
JOIN session_info
ON ep.ga_session_id = session_info.ga_session_id
GROUP BY
session_info.date,
 session_info.country,
 session_info.device,
 session_info.continent,
 session_info.channel,
 session_info.test,
 session_info.test_group,
 ep.event_name),


session AS (
SELECT
session_info.date,
 session_info.country,
 session_info.device,
 session_info.continent,
 session_info.channel,
 session_info.test,
 session_info.test_group,
 COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
FROM session_info
GROUP BY
 session_info.date,
 session_info.country,
 session_info.device,
 session_info.continent,
 session_info.channel,
 session_info.test,
 session_info.test_group),


account AS (
SELECT session_info.date,
 session_info.country,
 session_info.device,
 session_info.continent,
 session_info.channel,
 session_info.test,
 session_info.test_group,
 COUNT(DISTINCT acs.ga_session_id) AS new_account
 FROM `DA.account_session` acs
 JOIN session_info
 ON acs.ga_session_id = session_info.ga_session_id
 GROUP BY session_info.date,
 session_info.country,
 session_info.device,
 session_info.continent,
 session_info.channel,
 session_info.test,
 session_info.test_group)


SELECT session_with_orders.date,
 session_with_orders.country,
 session_with_orders.device,
 session_with_orders.continent,
 session_with_orders.channel,
 session_with_orders.test,
 session_with_orders.test_group,
 'session_with_orders' AS event_name,
 session_with_orders.session_with_orders AS value
FROM session_with_orders
UNION ALL
SELECT
events.date,
 events.country,
 events.device,
 events.continent,
 events.channel,
 events.test,
 events.test_group,
 event_name,
 events.event_cnt AS value
FROM events
UNION ALL
SELECT
session.date,
 session.country,
 session.device,
 session.continent,
 session.channel,
 session.test,
 session.test_group,
 'session' AS event_name,
 session_cnt AS value
FROM session
UNION ALL
SELECT
account.date,
 account.country,
 account.device,
 account.continent,
 account.channel,
 account.test,
 account.test_group,
 'new_account' AS event_name,
 new_account AS value
FROM account
"""

query_job = client.query(query)

df = query_job.result().to_dataframe(create_bqstorage_client=False)

df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,Ecuador,desktop,Americas,Organic Search,2,2,session_with_orders,1
1,2020-11-02,Trinidad & Tobago,desktop,Americas,Organic Search,2,2,session_with_orders,1
2,2020-11-03,Cambodia,desktop,Asia,Direct,2,1,session_with_orders,1
3,2020-11-03,Paraguay,desktop,Americas,Direct,2,2,session_with_orders,1
4,2020-11-04,Kazakhstan,desktop,Asia,Undefined,2,1,session_with_orders,1


In [ ]:
metrics = {
    'add_payment_info': 'add_payment_info',
    'add_shipping_info': 'add_shipping_info',
    'begin_checkout': 'begin_checkout',
    'new_accounts': 'new_account'
}

alpha = 0.05

In [ ]:
control_group = 1
experimental_group = 2

In [ ]:
tests = df['test'].dropna().unique()

tests

<IntegerArray>
[2, 1, 4, 3]
Length: 4, dtype: Int64

In [ ]:
results = []

for test in tests:

    test_data = df[df['test'] == test] #data for current test

    control_data = test_data[test_data['test_group'] == control_group] #control group

    experimental_data = test_data[test_data['test_group'] == experimental_group] #experimental group

    control_sessions = control_data.loc[control_data['event_name'] == 'session', 'value'].sum() #number of sessions

    experimental_sessions = experimental_data.loc[experimental_data['event_name'] == 'session', 'value'].sum() # number of sessions

    #calculate all metrics
    for metric_name, event_name in metrics.items():

        control_conversions = control_data.loc[control_data['event_name'] == event_name, 'value'].sum() #number of conversions

        experimental_conversions = experimental_data.loc[experimental_data['event_name'] == event_name, 'value'].sum() #number of conversions

        control_rate = control_conversions / control_sessions #conversion rate

        experimental_rate = experimental_conversions / experimental_sessions #conversion rate

        count = np.array([experimental_conversions, control_conversions]) #number of conversions

        nobs = np.array([experimental_sessions, control_sessions]) #number of sessions

        z_stat, p_value = proportions_ztest(count, nobs) #two-proportion z-test

        metric_change = (experimental_rate - control_rate) / control_rate * 100 #relative change

        significant = p_value < alpha

        #results
        results.append({
            'test_number': test,
            'metric': metric_name,
            'numerator_ev': experimental_conversions,
            'denominator_ev': experimental_sessions,
            'conversion_rate_ev': experimental_rate,
            'numerator_co': control_conversions,
            'denominator_co': control_sessions,
            'conversion_rate_co': control_rate,
            'metric_change': metric_change,
            'z_stat': z_stat,
            'p_value': p_value,
            'significant': significant
        })

In [ ]:
results_df = pd.DataFrame(results)

results_df

,test_number,metric,numerator_ev,denominator_ev,conversion_rate_ev,numerator_co,denominator_co,conversion_rate_co,metric_change,z_stat,p_value,significant
0,2,add_payment_info,2409,50244,0.047946,2344,50637,0.046290,3.576911,1.240994,0.214608,False
1,2,add_shipping_info,3510,50244,0.069859,3480,50637,0.068724,1.650995,0.709557,0.477979,False
2,2,begin_checkout,4313,50244,0.085841,4262,50637,0.084168,1.988164,0.952898,0.340642,False
3,2,new_accounts,4184,50244,0.083274,4165,50637,0.082252,1.241934,0.588793,0.556000,False
4,1,add_payment_info,2229,45193,0.049322,1988,45362,0.043825,12.542021,3.924884,0.000087,True
5,1,add_shipping_info,3221,45193,0.071272,3034,45362,0.066884,6.560481,2.603571,0.009226,True
6,1,begin_checkout,4021,45193,0.088974,3784,45362,0.083418,6.660587,2.978783,0.002894,True
7,1,new_accounts,3681,45193,0.081451,3823,45362,0.084278,-3.354299,-1.542883,0.122859,False
8,4,add_payment_info,3601,105141,0.034249,3731,105079,0.035507,-3.541234,-1.571106,0.116158,False
9,4,add_shipping_info,4956,105141,0.047137,5128,105079,0.048801,-3.411125,-1.785795,0.074132,False


In [ ]:
results_display = results_df.copy()

results_display['conversion_rate_ev'] = (results_display['conversion_rate_ev'] * 100).round(2)

results_display['conversion_rate_co'] = (results_display['conversion_rate_co'] * 100).round(2)

results_display['metric_change'] = results_display['metric_change'].round(2)

results_display['z_stat'] = results_display['z_stat'].round(4)

results_display['p_value'] = results_display['p_value'].round(6)

results_display

,test_number,metric,numerator_ev,denominator_ev,conversion_rate_ev,numerator_co,denominator_co,conversion_rate_co,metric_change,z_stat,p_value,significant
0,2,add_payment_info,2409,50244,4.79,2344,50637,4.63,3.58,1.2410,0.214608,False
1,2,add_shipping_info,3510,50244,6.99,3480,50637,6.87,1.65,0.7096,0.477979,False
2,2,begin_checkout,4313,50244,8.58,4262,50637,8.42,1.99,0.9529,0.340642,False
3,2,new_accounts,4184,50244,8.33,4165,50637,8.23,1.24,0.5888,0.556000,False
4,1,add_payment_info,2229,45193,4.93,1988,45362,4.38,12.54,3.9249,0.000087,True
5,1,add_shipping_info,3221,45193,7.13,3034,45362,6.69,6.56,2.6036,0.009226,True
6,1,begin_checkout,4021,45193,8.90,3784,45362,8.34,6.66,2.9788,0.002894,True
7,1,new_accounts,3681,45193,8.15,3823,45362,8.43,-3.35,-1.5429,0.122859,False
8,4,add_payment_info,3601,105141,3.42,3731,105079,3.55,-3.54,-1.5711,0.116158,False
9,4,add_shipping_info,4956,105141,4.71,5128,105079,4.88,-3.41,-1.7858,0.074132,False


In [ ]:
results_display.to_csv('ab_test_results.csv', index=False)

from google.colab import files
files.download('ab_test_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

| Test | add_payment_info | add_shipping_info | begin_checkout | new_accounts |
|---|---|---|---|---|
| 1 | significant, +12.54% | significant, +6.56% | significant, +6.66% | not significant (-3.35%) |
| 2 | not significant | not significant | not significant | not significant |
| 3 | not significant | not significant | significant, -3.35% | not significant |
| 4 | not significant | not significant | significant, -2.35% | significant, -3.36% |

- **Test 1** is the only test with a clear positive effect: three checkout-funnel metrics increased significantly, while new accounts did not change significantly.
- **Test 2** shows no significant change in any metric.
- **Test 3** significantly decreased begin_checkout (-3.35%); the other metrics did not change significantly.
- **Test 4** significantly decreased begin_checkout (-2.35%) and new_accounts (-3.36%).

**Recommendation:** roll out the variant from test 1. Reject the variants from tests 3 and 4, as they hurt key metrics. Test 2 is inconclusive and would need a larger sample or a different hypothesis.